In [ ]:
import os
import sys
from dotenv import load_dotenv

# 1. Enrutamiento del entorno relativo
root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.agent.qa_system import setup_qa_chain

load_dotenv()

print(">>> CONFIGURANDO MOTOR DE BÚSQUEDA CONVERSACIONAL RAG <<<")
# Inicializamos el motor cargando el índice FAISS generado en la Fase 2
qa_chain = setup_qa_chain(vectorstore_path="../vector_store/faiss_index")

# Variable global que actuará como memoria persistente de la sesión
historial_chat = [] 
print("Sistema RAG listo para recibir consultas interactiva.")

In [ ]:
print("🤖 CTBOT: ASISTENTE CONSULTECH")
print("Escribe 'salir' o 'exit' para dar por terminada la sesión analítica.\n" + "="*60)

while True:
    # Captura de pregunta en lenguaje natural desde la interfaz de Databricks
    pregunta_usuario = input("Tu consulta sobre los documentos: ")
    
    if pregunta_usuario.lower().strip() in ['salir', 'exit']:
        print(">> Sesión interactiva finalizada con éxito.")
        break
        
    if not pregunta_usuario.strip():
        continue
        
    # Invocación de la cadena pasando la pregunta y el estado de la memoria histórica
    respuesta = qa_chain.invoke({"question": pregunta_usuario, "chat_history": historial_chat})
    
    print(f"\nRespuesta del Consultor:\n{respuesta['answer']}")
    
    print("\nFuentes Citadas (Rúbrica de Criterio Experto):")
    fuentes_unicas = set()
    for doc in respuesta["source_documents"]:
        fuentes_unicas.add(doc.metadata.get('source', 'Origen Desconocido'))
        
    for fuente in fuentes_unicas:
        print(f" - Documento Fuente: {os.path.basename(fuente)}")
    print("="*60 + "\n")
    
    # Actualización estricta del contexto histórico para la siguiente iteración
    historial_chat.append((pregunta_usuario, respuesta["answer"]))